# The Shape of News

## A data-mining study of editorial categories, similarity structure, and articles that do not fit cleanly

**Author:** Anika Garg  
**UIN:** 330004874


## Executive Summary

News articles are usually presented as if each belongs to a single clean category: Sports, Finance, Health, Lifestyle, News, and so on. This project tests whether those editorial labels correspond to measurable structure in article text, and then asks what makes an article appear anomalous within that structure.

The central finding is that editorial categories are **real but porous**. Some categories, especially Sports and parts of Finance, have distinctive vocabularies. However, the full corpus does not separate into 15 clean category-shaped clusters. Topic models, K-Means, similarity graphs, and UMAP projections all suggest that MIND's labels combine at least three things: topical domain, editorial function, and recurring news-event vocabulary.

The original hypothesis was that outlier articles would be **bridge articles**: stories that sit between categories, such as Health-Lifestyle or Finance-News crossovers. The evidence points in a different direction. The articles flagged as anomalous are usually **peripheral within their assigned categories**, not clean bridges between categories. They tend to be longer, richer in named entities, and more specific than the category's typical short article. In other words, they are not weak articles; they are often articles with more texture than their label can comfortably summarize.


## Background and Research Questions

The MIND news dataset assigns each article to an editorial category. These labels are useful for navigation and recommendation, but they are not guaranteed to be the same thing as statistical clusters in text space. A category can be meaningful to readers even if its articles do not form a compact machine-learning cluster.

This distinction matters for recommender systems. If an article is far from its assigned category, the system needs to know whether it is truly off-topic, whether it bridges multiple reader interests, or whether it is simply a detailed article in a broad editorial bucket.

This notebook asks four connected questions:

1. **Q0: Are editorial categories visible in article text?**  
   Test whether categories have distinct vocabularies, whether topic models rediscover the editorial labels, and whether hard clustering aligns with those labels.

2. **Q1: Are category boundaries porous?**  
   Build a similarity graph to examine whether articles connect mostly within their categories or across category boundaries.

3. **Q2: Where do statistical outliers sit?**  
   Use Isolation Forest to identify anomalous articles, then test whether they are bridges between categories or peripheral cases within their own categories.

4. **Q3: What does an outlier actually look like?**  
   Use case studies, sentiment, and named-entity analysis to describe the qualitative signature of anomalous articles.

The analysis is exploratory rather than predictive: the goal is not to build the best classifier, but to understand what category structure means in a real news corpus.


---
## Setup environment


In [ ]:
!pip install -q "numpy==1.26.4" "pandas==2.2.2" "scipy==1.12.0"
!pip install -q scikit-learn networkx matplotlib seaborn
!pip install -q nltk umap-learn wordcloud textblob
!pip install -q gensim
!pip install -q node2vec
!pip install -q faiss-cpu
!pip install -q spacy
!python -m spacy download en_core_web_sm -q
print("Setup complete — run the imports cell next.")


In [ ]:
import json, re, time, warnings
from collections import Counter
from functools import reduce

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from sklearn.random_projection import SparseRandomProjection
from sklearn.preprocessing import LabelEncoder

from gensim import corpora
from gensim.models import LdaModel
from wordcloud import WordCloud
from textblob import TextBlob
import spacy

import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

import faiss
import umap as umap_lib

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 80)
SEED = 42
np.random.seed(SEED)
print("Libraries loaded.")

In [ ]:
!wget -q https://huggingface.co/datasets/yjw1029/MIND/resolve/main/MINDlarge_train.zip
!unzip -q -o MINDlarge_train.zip

news = pd.read_csv("MINDlarge_train/news.tsv", sep="\t", header=None)
news.columns = ["id", "category", "subcategory", "title", "abstract",
                "url", "entities_title", "entities_abstract"]

print(f"Loaded {len(news):,} articles")
print(f"\nCategory distribution:")
print(news['category'].value_counts().to_string())

## Data Pre-Processing

Before cleaning anything, look at what the raw dataset actually contains. This surfaces issues that bias downstream analysis if ignored: missing fields, label inconsistencies, length outliers, and the duplicate problem that is endemic to wire-service news data.


In [ ]:
print(f"Shape: {news.shape[0]:,} rows × {news.shape[1]} columns\n")

# Missing values
print("Missing values per column:")
print(news.isna().sum().to_string())

# Category label hygiene
raw_unique  = news['category'].nunique()
norm_unique = news['category'].str.strip().str.lower().nunique()
ws_issues   = (news['category'] != news['category'].str.strip()).sum()
print(f"\nCategory labels — unique raw: {raw_unique}, unique normalized: {norm_unique}")
print(f"Labels with leading/trailing whitespace: {ws_issues}")

# Length distributions
title_len = news['title'].fillna('').str.split().str.len()
abst_len  = news['abstract'].fillna('').str.split().str.len()
print(f"\nTitle words    — min {title_len.min()}, median {title_len.median():.0f}, max {title_len.max()}")
print(f"Abstract words — min {abst_len.min()}, median {abst_len.median():.0f}, max {abst_len.max()}")
print(f"Articles with empty abstract: {(abst_len == 0).sum():,} ({(abst_len == 0).mean():.1%})")

# Duplicates
exact_dupes = news.duplicated(subset=['title', 'abstract']).sum()
title_dupes = news.duplicated(subset=['title']).sum()
print(f"\nExact duplicates (title+abstract): {exact_dupes:,}")
print(f"Title-only duplicates (likely wire-service republishes): {title_dupes:,}")

# Tiny categories
cat_counts = news['category'].value_counts()
small_cats = cat_counts[cat_counts < 100]
print(f"\nCategories with <100 articles (clustering noise): {len(small_cats)}")
if len(small_cats):
    print(small_cats.to_string())


## Cleaning & Preprocessing

Cleaning steps:
1. Category normalization — strip whitespace, lowercase, drop categories with <100 articles.
2. **News-specific stopwords** — reporting verbs (*said, told, according*), weekdays/months, wire-service tokens (*reuters, getty, photo*) on top of standard English stopwords. These dominate news TF-IDF without distinguishing categories.
3. **spaCy lemmatization** — collapses inflections (*winning/wins/won → win*) so the similarity graph in Q1 does not fracture topics across surface forms.
4. **Length filtering** — keep articles between 5 and 500 tokens.
5. **Exact + near-duplicate removal** — drop exact title+abstract matches, then drop articles sharing a normalized title (catches the wire-service republishing pattern).
6. **Categorical dtype** — converts `category`/`subcategory` for memory + groupby speed.


In [ ]:
n_start = len(news)

# Step 1: Normalize category labels and drop tiny categories
news['category']    = news['category'].str.strip().str.lower()
news['subcategory'] = news['subcategory'].fillna('').str.strip().str.lower()

MIN_CATEGORY_SIZE = 100
cat_counts = news['category'].value_counts()
keep_cats  = cat_counts[cat_counts >= MIN_CATEGORY_SIZE].index
n_before = len(news)
news = news[news['category'].isin(keep_cats)].reset_index(drop=True)
print(f"[1] Tiny-category drop: removed {n_before - len(news):,} articles "
      f"in {len(cat_counts) - len(keep_cats)} small categories")

# Step 2: Build extended stopword set
NEWS_STOPWORDS = {
    # reporting verbs
    'said','says','told','asked','added','noted','reported','reports',
    'according','stated','announced','confirmed','explained','call','calls','called',
    # temporal
    'monday','tuesday','wednesday','thursday','friday','saturday','sunday',
    'january','february','march','april','may','june','july','august',
    'september','october','november','december',
    'today','yesterday','tomorrow','week','weeks','year','years',
    'month','months','day','days','morning','evening','night',
    # generic filler
    'new','news','first','last','also','people','time','times','way',
    'thing','things','get','got','going','make','made','take','took',
    'see','know','think','want','really','much','many','one','two','three',
    # wire/source/media artifacts
    'reuters','ap','cnn','bbc','photo','getty','image','images',
    'video','click','read','share','tweet','article','story',
}
stop_words_full = set(stopwords.words('english')) | NEWS_STOPWORDS
print(f"[2] Stopword set: {len(stop_words_full):,} words "
      f"(English + {len(NEWS_STOPWORDS)} news-specific)")

# Step 3: Combine title + abstract
news['title']    = news['title'].fillna('')
news['abstract'] = news['abstract'].fillna('')
news['text']     = (news['title'] + ' ' + news['abstract']).str.strip()

# Step 4: Lemmatize with spaCy
print(f"[3] Lemmatizing {len(news):,} articles (spaCy, ~1-3 min on Colab CPU)...")
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner', 'attribute_ruler'])

def normalize(doc):
    return ' '.join(
        tok.lemma_.lower() for tok in doc
        if tok.is_alpha
        and len(tok.lemma_) > 2
        and tok.lemma_.lower() not in stop_words_full
    )

t0 = time.time()
news['clean'] = [normalize(d) for d in nlp.pipe(news['text'].tolist(), batch_size=500)]
print(f"    done in {time.time() - t0:.1f}s")

# Step 5: Length filtering (min AND max)
news['text_length'] = news['clean'].str.split().str.len()
n_before = len(news)
news = news[news['text_length'].between(5, 500)]
print(f"[4] Length filter (5-500 tokens): dropped {n_before - len(news):,} articles")

# Step 6: Exact duplicate removal
n_before = len(news)
news = news.drop_duplicates(subset=['title', 'abstract'])
print(f"[5a] Exact-duplicate drop: removed {n_before - len(news):,} articles")

# Step 7: Near-duplicate detection via normalized title
# Catches wire-service republishes that differ only in punctuation/casing
news['_title_norm'] = (news['title'].str.lower()
                                     .str.replace(r'[^a-z0-9 ]', '', regex=True)
                                     .str.replace(r'\s+', ' ', regex=True)
                                     .str.strip())
n_before = len(news)
news = news.drop_duplicates(subset=['_title_norm'])
news = news.drop(columns=['_title_norm']).reset_index(drop=True)
print(f"[5b] Near-duplicate drop (normalized title): removed {n_before - len(news):,} articles")

# Step 8: Categorical dtype for memory + speed
news['category']    = news['category'].astype('category')
news['subcategory'] = news['subcategory'].astype('category')

# ── Summary ──────────────────────────────────────────────────────────
n_end = len(news)
print(f"\n{'='*60}")
print(f"Pipeline complete: {n_start:,} → {n_end:,} articles "
      f"({(n_start - n_end) / n_start:.1%} removed)")
print(f"Categories retained: {news['category'].nunique()}")
print(f"Memory: {news.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"{'='*60}")

news[['category', 'title', 'clean']].head(3)


In [ ]:
# ── Global TF-IDF representation ──────────────
vectorizer = TfidfVectorizer(max_features=8000, stop_words='english', sublinear_tf=True)
X = vectorizer.fit_transform(news['clean'])
terms = vectorizer.get_feature_names_out()

print(f"TF-IDF matrix: {X.shape[0]:,} articles × {X.shape[1]:,} features")
print(f"Sparsity: {1 - X.nnz/(X.shape[0]*X.shape[1]):.2%}")

# Q0: Do editorial categories have measurable textual structure?

Before asking which articles do not fit their categories, we need to establish that the categories carry textual signal at all. If the labels were unrelated to article language, then an "outlier within category" would be meaningless. Q0 therefore tests the basic premise in three ways: vocabulary distinctiveness, topic modeling, and hard clustering.


## Q0A: Do categories have distinct vocabularies?

The simplest possible version of "do the categories mean something." If Sports articles mostly use Sports words and Finance articles mostly use Finance words, we are in business. If every category uses the same top words, we would have to stop here.


In [ ]:
# Binary word presence per article (for frequency analysis)
cv = CountVectorizer(max_features=1000, stop_words='english', binary=True)
X_bin = cv.fit_transform(news['clean'])
freq_global = pd.DataFrame(X_bin.toarray(), columns=cv.get_feature_names_out()).sum().sort_values(ascending=False)

plt.figure(figsize=(12, 4))
freq_global.head(25).plot(kind='bar', color='steelblue')
plt.title("Top 25 Most Frequent Words Across All Articles", fontsize=13)
plt.ylabel("Articles containing word")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Per-category top words — the core vocabulary distinctiveness test
top_cats = news['category'].value_counts().head(6).index

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

category_top_words = {}  # save for later use in RQ1b

for ax, cat in zip(axes, top_cats):
    text  = ' '.join(news[news['category'] == cat]['clean'])
    common = Counter(text.split()).most_common(12)
    category_top_words[cat] = [w for w, _ in common]
    words, counts = zip(*common)
    ax.barh(words[::-1], counts[::-1], color='teal')
    ax.set_title(cat, fontsize=12, fontweight='bold')
    ax.set_xlabel("Frequency")

plt.suptitle("Top Words per Category — How Distinct Is Each Vocabulary?", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**Q0A takeaway.** Sports and Finance show clear vocabulary signatures: Sports articles emphasize terms such as game, team, and season, while Finance articles contain words tied to money, business, and local economic activity. Other categories are less cleanly separated. News, Lifestyle, Travel, Video, and Health share more general-purpose vocabulary, which suggests that some labels describe editorial packaging as much as topical content.


In [ ]:
# Bigrams: two-word phrases reveal more specific topic signals
bigram_cv = CountVectorizer(ngram_range=(2,2), max_features=500, stop_words='english')

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for ax, cat in zip(axes, top_cats):
    subset_text = news[news['category'] == cat]['clean']
    X_bg = bigram_cv.fit_transform(subset_text)
    bg_counts = X_bg.toarray().sum(axis=0)
    bg_words  = bigram_cv.get_feature_names_out()
    top_bg = sorted(zip(bg_words, bg_counts), key=lambda x: x[1], reverse=True)[:10]
    bw, bc = zip(*top_bg)
    ax.barh(bw[::-1], bc[::-1], color='darkorange')
    ax.set_title(f"{cat} — top bigrams", fontsize=11, fontweight='bold')

plt.suptitle("Top Bigrams per Category", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

The bigrams reinforce the unigram pattern. Sports bigrams remain highly domain-specific, while Finance bigrams point toward real estate and local-economy coverage rather than only markets or earnings. By contrast, News, Travel, Video, and Lifestyle share many generic phrases.

**Conclusion.** Category labels are not arbitrary, but they are uneven. Some categories act like compact topical domains; others are broad editorial containers that mix several kinds of articles.


In [ ]:
# Vocabulary overlap matrix: what fraction of a category's top-50 words
# are shared with each other category?
all_cats = sorted(news['category'].unique())
cat_vocab = {}
for cat in all_cats:
    text  = ' '.join(news[news['category'] == cat]['clean'])
    cat_vocab[cat] = set(w for w, _ in Counter(text.split()).most_common(50))

overlap = pd.DataFrame(index=all_cats, columns=all_cats, dtype=float)
for c1 in all_cats:
    for c2 in all_cats:
        shared = len(cat_vocab[c1] & cat_vocab[c2])
        overlap.loc[c1, c2] = shared / 50.0

plt.figure(figsize=(12, 9))
sns.heatmap(overlap.astype(float), annot=True, fmt='.0%', cmap='YlOrRd',
            linewidths=0.4, cbar_kws={'label': 'Shared fraction of top-50 words'})
plt.title("Vocabulary Overlap Between Categories\n(diagonal = 100%, off-diagonal = shared words)", fontsize=13)
plt.tight_layout()
plt.show()

print("Most overlapping pairs (excluding diagonal):")
flat = overlap.copy()
np.fill_diagonal(flat.values, 0)
flat_series = flat.stack().sort_values(ascending=False)
print(flat_series.head(10).to_string())

**Observation from the overlap matrix.**

The heaviest overlap occurs among broad lifestyle- and information-oriented categories. Travel and News share a large fraction of their top vocabulary, and Lifestyle, Health, Travel, and Food-and-Drink also overlap substantially. Sports and Finance overlap much less with other categories, which confirms that they are among the most lexically distinctive labels.

**Q0A conclusion.** Editorial labels contain real signal, but the signal is not equally strong across the taxonomy. This matters because anomaly detection will be easier and more interpretable in categories with tight vocabularies than in categories whose language naturally overlaps with several other categories.


## Q0B: Does a topic model rediscover the editorial categories?

LDA is a probabilistic topic model. It receives article text without category labels and estimates latent themes that best explain word co-occurrence patterns. This is a useful stress test for the editorial taxonomy: if the 15 categories were also the dominant latent themes in the corpus, an unsupervised topic model should tend to recover something close to them.

However, LDA topics and editorial categories are not the same object. LDA finds **latent word-co-occurrence themes**; editorial categories are **human-facing labels** designed for navigation, product organization, and reader expectations. A mismatch between the two is therefore not a failure of LDA or of the labels. It means the corpus has a different statistical structure than the user-facing taxonomy.


In [ ]:
# LDA: probabilistic topic model — bottom-up, no label information used
# We test multiple topic counts and pick the one with the best held-in log-perplexity
# score. Gensim returns a log bound, so less-negative / larger is better.

N_LDA = min(8000, len(news))
lda_idx = news.index[:N_LDA]
lda_sample = news.loc[lda_idx, 'clean'].fillna('')
texts_lda = [t.split() for t in lda_sample]

# Drop empty documents before building the dictionary/corpus.
nonempty_pairs = [(idx, toks) for idx, toks in zip(lda_idx, texts_lda) if toks]
lda_idx = pd.Index([idx for idx, _ in nonempty_pairs])
texts_lda = [toks for _, toks in nonempty_pairs]

if len(texts_lda) == 0:
    raise ValueError("No non-empty documents available for LDA after preprocessing.")

dictionary = corpora.Dictionary(texts_lda)
dictionary.filter_extremes(no_below=5, no_above=0.5)
corpus_bow = [dictionary.doc2bow(t) for t in texts_lda]

if len(dictionary) == 0:
    raise ValueError("LDA dictionary is empty after filter_extremes; lower no_below or inspect preprocessing.")

perplexities = []
# Avoid invalid k values if the vocabulary is unexpectedly small.
K_lda = [k for k in [5, 8, 10, 12, 15, 18, 20] if k <= max(2, len(dictionary))]

for k in K_lda:
    lda_tmp = LdaModel(corpus=corpus_bow, id2word=dictionary,
                       num_topics=k, passes=5, random_state=SEED)
    perplexities.append(lda_tmp.log_perplexity(corpus_bow))
    print(f"  k={k:2d}  log-perplexity={perplexities[-1]:.3f}")

best_k_lda = K_lda[perplexities.index(max(perplexities))]
print(f"\n→ Best LDA k: {best_k_lda}  (editorial categories: {news['category'].nunique()})")


In [ ]:
# Fit the best LDA model
lda = LdaModel(corpus=corpus_bow, id2word=dictionary,
               num_topics=best_k_lda, passes=15, random_state=SEED)

print("=== LDA Topics ===")
for i, topic in lda.print_topics(num_words=10):
    print(f"  Topic {i:2d}: {topic}")

In [ ]:
# Assign dominant LDA topic to each article in the LDA sample
doc_topics = [lda.get_document_topics(doc) for doc in corpus_bow]
dominant = [max(d, key=lambda x: x[1])[0] if d else np.nan for d in doc_topics]
news.loc[lda_idx, 'lda_topic'] = dominant

# How well do LDA topics align with editorial categories?
lda_view = news.loc[lda_idx].dropna(subset=['lda_topic']).copy()
lda_view['lda_topic'] = lda_view['lda_topic'].astype(int)
ct_lda = pd.crosstab(lda_view['category'], lda_view['lda_topic'])
ct_lda_norm = ct_lda.div(ct_lda.sum(axis=1), axis=0)

plt.figure(figsize=(14, 7))
sns.heatmap(ct_lda_norm, cmap='Blues', linewidths=0.3,
            cbar_kws={'label': 'Fraction of category in LDA topic'})
plt.title(f"LDA Topics ({best_k_lda}) vs Editorial Categories\n"
          "Each row sums to 1.0 — dark column = that LDA topic dominates this category", fontsize=13)
plt.xlabel("LDA Topic")
plt.ylabel("Editorial Category")
plt.tight_layout()
plt.show()


In [ ]:
# Word clouds for the top LDA topics
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i in range(min(8, best_k_lda)):
    top_words = dict(lda.show_topic(i, topn=50))
    wc = WordCloud(width=400, height=250, background_color='white',
                   prefer_horizontal=0.9).generate_from_frequencies(top_words)
    axes[i].imshow(wc, interpolation='bilinear')
    axes[i].axis('off')
    axes[i].set_title(f"LDA Topic {i}", fontsize=10)

for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.suptitle("Word Clouds — What Is Each LDA Topic About?", fontsize=14)
plt.tight_layout()
plt.show()

**Q0B takeaway.** LDA's perplexity sweep prefers a much smaller number of topics than the number of editorial categories. The topics that emerge correspond to the most lexically distinctive slices of the corpus, while several editorial categories collapse into broader themes.

This should be interpreted carefully. LDA is not proving that the editorial labels are wrong. Instead, it shows that the strongest latent word-co-occurrence themes are coarser than the category taxonomy. Editorial categories encode human/product distinctions that are finer, broader, or more contextual than what a bag-of-words topic model can recover.


## Q0C: Does hard clustering rediscover the categories?

K-Means asks a different version of the same question. Instead of estimating probabilistic topics, it partitions TF-IDF vectors into hard clusters. If editorial categories were compact regions in text space, K-Means clusters should align strongly with the labels. We evaluate that alignment using Adjusted Rand Index (ARI) and per-category purity.


In [ ]:
# Silhouette sweep to find natural cluster count
sil_scores = []
evaluated_K = []
K_range = range(2, 21)
N_SIL = min(5000, X.shape[0])
idx_sil = np.random.choice(X.shape[0], N_SIL, replace=False)
X_sil = X[idx_sil]

for k in K_range:
    if k >= N_SIL:
        continue
    mbk = MiniBatchKMeans(n_clusters=k, random_state=SEED, batch_size=2000, n_init=5)
    labels = mbk.fit_predict(X_sil)
    sil_sample_size = min(2000, N_SIL)
    score = silhouette_score(X_sil, labels, sample_size=sil_sample_size, random_state=SEED)
    evaluated_K.append(k)
    sil_scores.append(score)
    print(f"  k={k:2d}  silhouette={score:.4f}")

best_k = evaluated_K[sil_scores.index(max(sil_scores))]
print(f"\n→ Data-preferred k={best_k}  |  Editorial categories={news['category'].nunique()}  |  Best LDA k={best_k_lda}")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(K_range, sil_scores, marker='o', color='steelblue', linewidth=2)
ax.axvline(best_k, color='tomato', linestyle='--', linewidth=1.5, label=f'K-Means preferred k={best_k}')
ax.axvline(best_k_lda, color='darkorange', linestyle='--', linewidth=1.5, label=f'LDA preferred k={best_k_lda}')
ax.axvline(news['category'].nunique(), color='gray', linestyle=':', linewidth=1.5,
           label=f'Editorial categories={news["category"].nunique()}')
ax.set_xlabel("Number of Topics / Clusters", fontsize=12)
ax.set_ylabel("Silhouette Score", fontsize=12)
ax.set_title("How Many Topics Does the Corpus Prefer?\nComparing K-Means, LDA, and Editorial Labels", fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Fit final K-Means and measure ARI
mbk_best    = MiniBatchKMeans(n_clusters=best_k, random_state=SEED, batch_size=2000, n_init=10)
news['cluster'] = mbk_best.fit_predict(X)

le = LabelEncoder()
news['cat_int'] = le.fit_transform(news['category'])

ari = adjusted_rand_score(news['cat_int'], news['cluster'])
print(f"Adjusted Rand Index (K-Means clusters vs editorial labels): {ari:.4f}")
print(f"  0 = random agreement, 1 = perfect agreement")

In [ ]:
# Per-category purity
purity_rows = []
for cat in news['category'].unique():
    sub = news[news['category'] == cat]
    dom = sub['cluster'].mode()[0]
    p   = (sub['cluster'] == dom).mean()
    purity_rows.append({'Category': cat, 'Articles': len(sub),
                        'Purity': round(p, 3)})
purity_df = pd.DataFrame(purity_rows).sort_values('Purity', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2ecc71' if p >= 0.70 else '#e67e22' if p >= 0.45 else '#e74c3c'
          for p in purity_df['Purity']]
ax.barh(purity_df['Category'], purity_df['Purity'], color=colors)
ax.axvline(0.70, color='green',  linestyle='--', alpha=0.6, label='Well-recovered (≥0.70)')
ax.axvline(0.45, color='orange', linestyle='--', alpha=0.6, label='Partially-recovered (≥0.45)')
ax.set_xlabel("Cluster Purity", fontsize=12)
ax.set_title("How Well Does Text Clustering Recover Each Editorial Category?", fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()
print(purity_df.to_string(index=False))

**Q0C takeaway.** K-Means and LDA identify structure at different levels of granularity. K-Means prefers a larger number of clusters, while LDA prefers fewer broad themes. Yet the Adjusted Rand Index remains close to zero, meaning that the K-Means partition does not map cleanly onto the editorial labels.

The important interpretation is that categories are **multi-dimensional**, not absent. A label may reflect topic, format, event type, audience intent, and editorial convention all at once. TF-IDF clustering captures only part of that structure, so it should not be expected to perfectly reconstruct a human-designed taxonomy.

This result motivates the next section: instead of asking whether categories form isolated clusters, Q1 asks whether they form a porous similarity network.


# Q1: Are articles that do not fit actually bridges between categories?

The initial hypothesis is that anomalous articles are **crossover articles**: stories that genuinely live between categories. For example, a health policy article might sit between Health and News, or a sports business article might sit between Sports and Finance.

If this hypothesis is correct, anomalous articles should occupy bridge positions in the similarity graph. They should have many neighbors from other categories, and they should sit in mixed regions rather than on the outer edge of their assigned category.


## Q1A: Building a similarity graph with FAISS

At nearly 100,000 articles, computing all pairwise cosine similarities is expensive. FAISS provides efficient nearest-neighbor search, so we use it to build a top-k similarity graph on a tractable sample of articles. Each node is an article; each edge links articles with similar TF-IDF representations.

This graph gives us a way to ask whether category boundaries behave like walls or membranes. If categories are cleanly separated, most edges should remain within the same category. If categories are porous, many edges should cross category labels.


In [ ]:
# Build a large top-k graph using FAISS for speed
N_GRAPH = min(8000, X.shape[0])
top_k = min(7, N_GRAPH - 1)
idx_graph = np.random.choice(X.shape[0], N_GRAPH, replace=False)
X_graph = X[idx_graph].toarray().astype('float32')
faiss.normalize_L2(X_graph)

fi = faiss.IndexFlatIP(X_graph.shape[1])
fi.add(X_graph)
_, neighbors = fi.search(X_graph, top_k + 1)

G = nx.Graph()
for local_i, row in enumerate(neighbors):
    orig_i = idx_graph[local_i]
    G.add_node(local_i, article_idx=orig_i,
               category=news.iloc[orig_i]['category'],
               cluster=int(news.iloc[orig_i]['cluster']))
    for local_j in row[1:]:
        if local_j != local_i and local_j >= 0:
            G.add_edge(local_i, int(local_j))

print(f"Graph — {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
print(f"Average degree: {sum(dict(G.degree()).values()) / max(1, G.number_of_nodes()):.1f}")


In [ ]:
# FAISS speed benchmark vs brute-force cosine
# IndexFlatIP is exact inner-product search; here it is faster because FAISS is optimized.
X_bench = X[:min(1000, X.shape[0])].toarray().astype('float32')
faiss.normalize_L2(X_bench)
bench_idx = faiss.IndexFlatIP(X_bench.shape[1])
bench_idx.add(X_bench)

t0 = time.time(); _ = cosine_similarity(X[:X_bench.shape[0]]); cos_time = time.time() - t0
t0 = time.time(); bench_idx.search(X_bench, min(5, X_bench.shape[0])); faiss_time = time.time() - t0

speedup = cos_time / faiss_time if faiss_time > 0 else np.inf
print(f"Brute-force cosine: {cos_time:.3f}s")
print(f"FAISS exact search: {faiss_time:.3f}s  ({speedup:.0f}× speedup)")

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['Cosine\n(brute-force)', 'FAISS\n(exact index)'], [cos_time, faiss_time],
       color=['#e74c3c', '#2ecc71'])
ax.set_title("Similarity Search Speed: FAISS vs Cosine")
ax.set_ylabel("Time (seconds)")
plt.tight_layout()
plt.show()


In [ ]:
# Community detection
from networkx.algorithms.community import greedy_modularity_communities
communities    = list(greedy_modularity_communities(G))
community_map  = {node: i for i, com in enumerate(communities) for node in com}
nx.set_node_attributes(G, community_map, 'community')

print(f"Communities detected: {len(communities)}")
print(f"Modularity: {nx.community.modularity(G, communities):.4f}")
print(f"Largest community sizes: {sorted([len(c) for c in communities], reverse=True)[:8]}")

In [ ]:
# UMAP projection
umap_model = umap_lib.UMAP(n_components=2, random_state=SEED, n_neighbors=15, min_dist=0.1)
X_umap     = umap_model.fit_transform(X_graph)

cats      = sorted(news['category'].unique())
palette   = plt.cm.get_cmap('tab20', len(cats))
cat_color = {c: palette(i) for i, c in enumerate(cats)}

node_cats   = [G.nodes[n]['category'] for n in G.nodes()]
node_colors = [cat_color[c] for c in node_cats]

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Left: by editorial category
axes[0].scatter(X_umap[:, 0], X_umap[:, 1], c=node_colors, s=6, alpha=0.6)
handles = [mpatches.Patch(color=cat_color[c], label=c) for c in cats]
axes[0].legend(handles=handles, title='Editorial Category',
               bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=7)
axes[0].set_title("By Editorial Category", fontsize=12)
axes[0].set_xlabel("UMAP-1"); axes[0].set_ylabel("UMAP-2")

# Right: by detected community
comm_colors = [community_map[n] for n in G.nodes()]
sc = axes[1].scatter(X_umap[:, 0], X_umap[:, 1], c=comm_colors,
                     cmap='tab20', s=6, alpha=0.6)
axes[1].set_title("By Graph Community (modularity clustering)", fontsize=12)
axes[1].set_xlabel("UMAP-1"); axes[1].set_ylabel("UMAP-2")
plt.colorbar(sc, ax=axes[1], label='Community ID')

plt.suptitle("UMAP Projection — Editorial Labels vs. Graph Communities", fontsize=14)
plt.tight_layout()
plt.show()

The UMAP projections show limited visual separation. When colored by editorial category, the articles form one diffuse cloud rather than separate category-specific islands. When colored by graph community, the structure is also mixed: communities exist in the modularity sense, but they are not cleanly visible as isolated 2D regions.

This is a central result. Editorial categories in MIND do not form clean geometric regions in TF-IDF + UMAP space. Therefore, an article that "does not fit" is unlikely to appear as a point sitting neatly between well-separated category clusters. The category space is already mixed.

The next step is to quantify this porousness directly through cross-category graph edges.


In [ ]:
# Cross-category edge porousness matrix
cross_edges = []
for u, v in G.edges():
    cu, cv = G.nodes[u]['category'], G.nodes[v]['category']
    if cu != cv:
        cross_edges.append(tuple(sorted([cu, cv])))

cross_counts = Counter(cross_edges)

all_cats_list = sorted(news['category'].unique())
cat_idx       = {c: i for i, c in enumerate(all_cats_list)}
n_c           = len(all_cats_list)
overlap_mat   = np.zeros((n_c, n_c))

for (ca, cb), cnt in cross_counts.items():
    i, j = cat_idx[ca], cat_idx[cb]
    overlap_mat[i, j] += cnt
    overlap_mat[j, i] += cnt

row_totals = overlap_mat.sum(axis=1, keepdims=True)
row_totals[row_totals == 0] = 1
overlap_norm = overlap_mat / row_totals

plt.figure(figsize=(13, 10))
sns.heatmap(overlap_norm, xticklabels=all_cats_list, yticklabels=all_cats_list,
            cmap='YlOrRd', linewidths=0.3,
            cbar_kws={'label': 'Fraction of cross-category edges'})
plt.title("Topic Porousness: How Much Do Categories Share Textual Neighbours?", fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

The porousness matrix gives a more detailed view than the vocabulary-overlap matrix. Many categories send a large fraction of their cross-category neighbors into News. Part of this is a size effect: News and Sports are large categories, so they naturally absorb more nearest-neighbor links. But the pattern is still meaningful because it shows that broad editorial categories function as hubs in the similarity graph.

**Q1A conclusion.** Category boundaries are porous. This supports the possibility of bridge articles, but it also warns us that cross-category edges alone are not enough to define an anomaly. In a porous taxonomy, many ordinary articles already have neighbors outside their assigned label.


## Q1B: Node2Vec — does graph structure tell a different story than TF-IDF?

Node2Vec learns vector embeddings of graph nodes via random walks. Two articles end up with similar embeddings if they live in similar positions in the graph — not necessarily because they share vocabulary, but because they share neighbors.

Why might this matter for our question? If "does not fit" articles are genuine crossovers, they might have a *structural* signature (they'd be in walked-through positions) that TF-IDF alone does not capture.


In [ ]:
from node2vec import Node2Vec

n2v_model_obj = Node2Vec(G, dimensions=64, walk_length=10,
                         num_walks=50, workers=1, quiet=True, seed=SEED)
model_n2v     = n2v_model_obj.fit(window=5, min_count=1)

embeddings = np.array([model_n2v.wv[str(i)] for i in range(len(idx_graph))])
print(f"Node2Vec embeddings: {embeddings.shape}")


In [ ]:
# Project Node2Vec embeddings to 2D and compare to TF-IDF UMAP
X_n2v_2d = umap_lib.UMAP(n_components=2, random_state=SEED).fit_transform(embeddings)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

axes[0].scatter(X_umap[:, 0], X_umap[:, 1], c=node_colors, s=6, alpha=0.6)
axes[0].set_title("TF-IDF UMAP\n(article content similarity)", fontsize=12)
axes[0].set_xlabel("UMAP-1"); axes[0].set_ylabel("UMAP-2")

axes[1].scatter(X_n2v_2d[:, 0], X_n2v_2d[:, 1], c=node_colors, s=6, alpha=0.6)
handles = [mpatches.Patch(color=cat_color[c], label=c) for c in cats]
axes[1].legend(handles=handles, title='Category', bbox_to_anchor=(1.01, 1),
               loc='upper left', fontsize=7)
axes[1].set_title("Node2Vec UMAP\n(graph-structural similarity)", fontsize=12)
axes[1].set_xlabel("UMAP-1"); axes[1].set_ylabel("UMAP-2")

plt.suptitle("TF-IDF Content Similarity vs. Node2Vec Graph Structure", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side retrieval comparison: same query, two methods
def tfidf_similar(query_local, topn=5):
    q_orig = idx_graph[query_local]
    sims   = cosine_similarity(X[q_orig], X[idx_graph]).flatten()
    top    = sims.argsort()[-topn-1:-1][::-1]
    return [(sims[i], news.iloc[idx_graph[i]]['category'], news.iloc[idx_graph[i]]['title']) for i in top]

def n2v_similar(query_local, topn=5):
    similar = model_n2v.wv.most_similar(str(query_local), topn=topn)
    return [(score, news.iloc[idx_graph[int(s)]]['category'],
             news.iloc[idx_graph[int(s)]]['title']) for s, score in similar]

for q in [0, 50, 200]:
    orig = idx_graph[q]
    print(f"\nQuery: [{news.iloc[orig]['category']}] {news.iloc[orig]['title'][:60]}")
    print(f"{'':>3}  TF-IDF results:")
    for score, cat, title in tfidf_similar(q):
        print(f"       [{cat:<14}] {title[:50]}  ({score:.3f})")
    print(f"{'':>3}  Node2Vec results:")
    for score, cat, title in n2v_similar(q):
        print(f"       [{cat:<14}] {title[:50]}  ({score:.3f})")

The retrieval comparison shows that TF-IDF and Node2Vec mostly agree for tightly defined categories, but diverge in broad, porous areas such as News, Lifestyle, and Health. TF-IDF retrieves articles with similar vocabulary; Node2Vec retrieves articles that occupy similar positions in the neighbor graph.

**Q1B conclusion.** Graph position adds useful context, especially where editorial labels overlap. However, this still does not prove that outliers are bridge articles. To test that claim, we need to explicitly identify anomalous articles and examine where they fall in the graph.


## Q1C: LSH — does the graph structure survive compression?

One sanity check before we move on. LSH (locality-sensitive hashing) compresses each article to a 128-bit binary fingerprint. If the communities we detected with the full graph survive this lossy compression, the structure is robust enough to use in a production recommender. If they do not survive, the structure we found is a fragile artifact.


In [ ]:
# LSH: random projection → binary hash → fast Hamming similarity
n_components = 128
rp = SparseRandomProjection(n_components=n_components, random_state=SEED)
X_proj = rp.fit_transform(X)

# SparseRandomProjection may return either a sparse matrix or a dense ndarray,
# depending on sklearn/scipy versions. Convert safely in both cases.
X_proj_arr = X_proj.toarray() if hasattr(X_proj, 'toarray') else np.asarray(X_proj)
X_hash = (X_proj_arr > 0).astype(np.uint8)

print(f"Each article compressed to a {n_components}-bit binary hash")


In [ ]:
# LSH speed vs exact cosine
t0 = time.time(); _ = cosine_similarity(X[0], X); exact_time = time.time() - t0
t0 = time.time(); _ = (X_hash == X_hash[0]).sum(axis=1); lsh_time = time.time() - t0

print(f"Exact cosine search: {exact_time:.4f}s")
print(f"LSH hash search:     {lsh_time:.4f}s  ({exact_time/lsh_time:.1f}× speedup)")

In [ ]:
# Key question: does LSH preserve the community structure we found in the full graph?
# For each graph community, measure what fraction of LSH neighbours stay in the same community.

retention_scores = []
sample_nodes = list(G.nodes())[:min(500, G.number_of_nodes())]

for node in sample_nodes:
    node_hash = X_hash[idx_graph[node]]
    # Find top-k LSH neighbours within the graph sample, excluding self.
    ham_sim = (X_hash[idx_graph] == node_hash).sum(axis=1)
    ranked = ham_sim.argsort()[::-1]
    lsh_neighbours = [int(nb) for nb in ranked if int(nb) != node][:top_k]

    my_community = community_map[node]
    same_community = sum(1 for nb in lsh_neighbours
                         if community_map.get(nb) == my_community)
    if len(lsh_neighbours):
        retention_scores.append(same_community / len(lsh_neighbours))

print("LSH community retention (fraction of LSH neighbours in same graph community):")
print(f"  Mean:   {np.mean(retention_scores):.3f}")
print(f"  Median: {np.median(retention_scores):.3f}")

plt.figure(figsize=(8, 4))
plt.hist(retention_scores, bins=20, color='steelblue', edgecolor='white')
plt.xlabel("Fraction of LSH neighbours in same community")
plt.ylabel("Number of articles")
plt.title("Does LSH Preserve Graph Community Structure?\n"
          "(1.0 = all neighbours in same community as exact graph)")
plt.axvline(np.mean(retention_scores), color='tomato', linestyle='--',
            label=f'Mean = {np.mean(retention_scores):.2f}')
plt.legend()
plt.tight_layout()
plt.show()


LSH retention is **moderate**, not high. The mean fraction of LSH neighbors that fall in the same graph community is meaningfully above random, but below what would be needed for a high-fidelity replacement of the full graph. The structure partially survives 128-bit compression, with real degradation.

**Q1C conclusion.** LSH is useful as a fast approximation for first-stage retrieval, but it should not replace the full similarity graph when the research question depends on subtle category-boundary structure.

---

### Q1 wrap-up

Q1 establishes that the category space is porous. Similarity links often cross labels, and graph embeddings reveal structure that is not identical to raw TF-IDF similarity. This gives the crossover hypothesis a plausible foundation.

The decisive test comes next: if outliers are true crossovers, they should have many cross-category neighbors. If they are peripheral articles within their own categories, they should instead sit farther from category centers without necessarily acting as bridges.


# Q2: Where do outliers actually sit?

Q2 tests the central hypothesis directly. We first identify statistical outliers using Isolation Forest, then locate those articles in the graph and UMAP space.

There are two competing explanations:

1. **Bridge hypothesis:** outliers are crossover articles with many neighbors in other categories.
2. **Peripheral-within-category hypothesis:** outliers are unusual relative to their assigned category but do not necessarily connect categories.

The distinction matters. A bridge article could improve recommendations across interests; a peripheral article may simply need richer category metadata or a secondary tag.


## Q2A: Isolation Forest — finding outliers, then locating them

Isolation Forest is a classic anomaly detector. It builds random trees and asks how quickly each point gets isolated — points that take few splits to isolate are anomalies. We run it on the TF-IDF features so we are asking "unusual in word-use space", then check where those anomalies land.


In [ ]:
N_ISO = min(10000, X.shape[0])
idx_iso = np.random.choice(X.shape[0], N_ISO, replace=False)
X_iso = X[idx_iso]

iso = IsolationForest(contamination=0.05, random_state=SEED, n_jobs=-1)
iso_labels = iso.fit_predict(X_iso)    # -1 = anomaly
iso_scores = iso.decision_function(X_iso)

news.loc[idx_iso, 'anomaly'] = iso_labels
news.loc[idx_iso, 'anomaly_score'] = iso_scores

iso_news = news.loc[idx_iso].copy()
iso_news['is_anomaly'] = iso_news['anomaly'] == -1

n_anom = (iso_labels == -1).sum()
print(f"Detected {n_anom:,} anomalies ({n_anom/N_ISO:.1%}) in {N_ISO:,} articles")


In [ ]:
# Anomalies on UMAP — are they at the edges of their category blobs?
graph_orig_idx = {idx_graph[n]: n for n in G.nodes()}

shared_rows = []
for iso_local, orig_idx in enumerate(idx_iso):
    if orig_idx in graph_orig_idx:
        gn = graph_orig_idx[orig_idx]
        shared_rows.append({'orig_idx': orig_idx, 'graph_node': gn,
                             'anomaly_score': iso_scores[iso_local],
                             'is_anomaly':    iso_labels[iso_local] == -1,
                             'category':      news.iloc[orig_idx]['category']})

shared_df = pd.DataFrame(shared_rows)
print(f"Articles in both graph and iso samples: {len(shared_df)}")

is_anomaly_umap = np.zeros(N_GRAPH, dtype=bool)
anom_score_umap = np.full(N_GRAPH, np.nan)
for row in shared_df.itertuples():
    if 0 <= row.graph_node < N_GRAPH:
        is_anomaly_umap[row.graph_node] = row.is_anomaly
        anom_score_umap[row.graph_node] = row.anomaly_score

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].scatter(X_umap[~is_anomaly_umap, 0], X_umap[~is_anomaly_umap, 1],
                c='lightgray', s=4, alpha=0.4, label='Normal')
axes[0].scatter(X_umap[is_anomaly_umap, 0],  X_umap[is_anomaly_umap, 1],
                c='tomato', s=20, alpha=0.85, label='Anomaly', zorder=3)
axes[0].set_title("Anomalies Highlighted in Article Space", fontsize=12)
axes[0].legend(); axes[0].set_xlabel("UMAP-1"); axes[0].set_ylabel("UMAP-2")

has_score = ~np.isnan(anom_score_umap)
sc2 = axes[1].scatter(X_umap[has_score, 0], X_umap[has_score, 1],
                       c=anom_score_umap[has_score],
                       cmap='coolwarm_r', s=10, alpha=0.8)
plt.colorbar(sc2, ax=axes[1], label='Anomaly Score (lower = more anomalous)')
axes[1].set_title("Continuous Anomaly Score on UMAP", fontsize=12)
axes[1].set_xlabel("UMAP-1"); axes[1].set_ylabel("UMAP-2")

plt.suptitle("Where Are Anomalous Articles in the Article Space?", fontsize=14)
plt.tight_layout()
plt.show()

The UMAP visualization shows that anomaly points are spread throughout the article cloud, with a mild tendency toward sparser outer regions. The continuous anomaly-score view is more informative than the binary flag: the most anomalous articles tend to appear toward the periphery rather than in obvious gaps between categories.

**Preliminary interpretation.** The visual evidence weakens the bridge hypothesis. If anomalies were primarily cross-category bridges, we would expect them to concentrate in boundary regions. Instead, they look more like category-edge cases.


In [ ]:
# Distance to category centroid — are anomalies peripheral?
cat_centroids = {}
for n in G.nodes():
    cat = G.nodes[n]['category']
    cat_centroids.setdefault(cat, []).append(X_umap[n])
cat_centroids = {c: np.mean(pts, axis=0) for c, pts in cat_centroids.items()}

dists, is_ano_list = [], []
for n in G.nodes():
    cat = G.nodes[n]['category']
    d   = np.linalg.norm(X_umap[n] - cat_centroids[cat])
    dists.append(d)
    is_ano_list.append(is_anomaly_umap[n])

dist_df = pd.DataFrame({'dist': dists, 'is_anomaly': is_ano_list})
norm_d = dist_df[~dist_df['is_anomaly']]['dist']
ano_d  = dist_df[dist_df['is_anomaly']]['dist']

print(f"Mean distance to category centroid:")
print(f"  Normal:   {norm_d.mean():.4f}")
print(f"  Anomaly:  {ano_d.mean():.4f}  ({ano_d.mean()/norm_d.mean():.2f}× farther)")

plt.figure(figsize=(9, 5))
plt.hist(norm_d, bins=40, alpha=0.6, density=True, color='steelblue',
         label=f'Normal (mean={norm_d.mean():.2f})')
plt.hist(ano_d,  bins=40, alpha=0.6, density=True, color='tomato',
         label=f'Anomaly (mean={ano_d.mean():.2f})')
plt.xlabel("Distance to Category Centroid in UMAP Space")
plt.ylabel("Density")
plt.title("Anomalous Articles Are Farther from Their Category's Centre")
plt.legend()
plt.tight_layout()
plt.show()

Anomalies are farther from their own category centers than normal articles are. The gap is modest, but it points in a consistent direction: anomalous articles tend to sit near the edges of their assigned categories.

**Q2A conclusion.** The evidence favors the peripheral-within-category hypothesis over the bridge hypothesis. Outliers are not mainly articles that connect two cleanly separated categories; they are articles whose vocabulary, length, entity density, or specificity makes them atypical for their assigned editorial bucket.


## Q2B: Do incoherent categories produce more outliers?

We have a per-category coherence score from Q0C (cluster purity). We have a per-category anomaly rate from Q2A. Is there a relationship?

Prediction: if outliers are peripheral (far from their category centroid), then categories whose centroid is poorly defined in the first place should produce more peripheral articles. Low-purity categories → more outliers.


In [ ]:
# Anomaly rate per category, plotted against cluster purity from RQ1
purity_lookup = purity_df.set_index('Category')['Purity']
anom_by_cat = iso_news.groupby('category', observed=False)[['is_anomaly']].agg(['mean', 'count'])
anom_by_cat.columns = ['Anomaly Rate', 'Total']

# Coerce to float: groupby on a categorical column gives a CategoricalIndex,
# and .index.map() can propagate that dtype, which breaks .min()/.max() later.
anom_by_cat['Cluster Purity'] = pd.to_numeric(anom_by_cat.index.map(purity_lookup), errors='coerce')

fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(anom_by_cat['Cluster Purity'], anom_by_cat['Anomaly Rate'],
                s=anom_by_cat['Total'] / 5,
                c=anom_by_cat['Anomaly Rate'], cmap='coolwarm_r',
                edgecolors='gray', linewidths=0.5, alpha=0.85)
plt.colorbar(sc, ax=ax, label='Anomaly Rate')

for cat, row in anom_by_cat.iterrows():
    ax.annotate(cat, (row['Cluster Purity'], row['Anomaly Rate']),
                fontsize=8, xytext=(4, 2), textcoords='offset points')

# Trend line, only when there are enough valid category points.
mask = anom_by_cat[['Cluster Purity', 'Anomaly Rate']].notna().all(axis=1)
x = anom_by_cat.loc[mask, 'Cluster Purity'].to_numpy(dtype=float)
y = anom_by_cat.loc[mask, 'Anomaly Rate'].to_numpy(dtype=float)
if len(x) >= 2:
    m, b = np.polyfit(x, y, 1)
    xs = np.linspace(x.min(), x.max(), 100)
    ax.plot(xs, m * xs + b, 'k--', alpha=0.4, label=f'Trend (slope={m:.2f})')
    ax.legend()

ax.set_xlabel("Cluster Purity (RQ1) — how well text clustering recovers this category", fontsize=11)
ax.set_ylabel("Anomaly Rate", fontsize=11)
ax.set_title("Q2B: Do Low-Purity Categories Produce More Anomalies?", fontsize=13)
plt.tight_layout()
plt.show()

if len(x) >= 2:
    print(f"Pearson r (purity vs anomaly rate): {np.corrcoef(x, y)[0, 1]:.4f}")
else:
    print("Not enough valid category points to compute Pearson correlation.")


The category-level relationship is consistent with the peripheral interpretation. Categories with lower cluster purity tend to have higher anomaly rates, while more coherent categories tend to produce fewer outliers.

**Q2B conclusion.** Outlier frequency is partly a property of the category, not just the individual article. Broad or internally mixed categories naturally create more edge cases because their centroids are less stable and their language is less uniform.

Together, Q2 refutes the strongest version of the crossover hypothesis. The more accurate model is that anomalous articles are often **over-specific, entity-rich, or texturally unusual examples inside broad editorial categories**.


# Q3: What does an outlier actually look like?

The quantitative results suggest that anomalies are peripheral rather than bridge-like. Q3 translates that result back into article-level evidence. We examine the most anomalous articles, compare one outlier to a typical article from the same category, and measure whether anomalies differ in sentiment and named-entity density.


## Q3A: The most anomalous articles — a first look

We will list the 15 most anomalous articles by Isolation Forest score.


In [ ]:
# What do the most anomalous articles look like?
print("The 15 most anomalous articles:")
print(f"{'Score':>8}  {'Category':<18}  {'Length':>6}  Title")
print("-" * 90)
for _, row in iso_news.nsmallest(15, 'anomaly_score').iterrows():
    print(f"{row['anomaly_score']:>8.4f}  {row['category']:<18}  "
          f"{int(row['text_length']):>6}  {str(row['title'])[:50]}")

The most anomalous articles are not random noise. They span several categories and often have specific, information-rich titles. This is already a useful correction to the original intuition: anomaly detection is not simply finding bad or empty articles. It is finding articles that have unusual textual profiles for their assigned categories.

**Q3A transition.** To make this concrete, the next cell compares the most anomalous article with a typical article from the same editorial category.


In [ ]:
# Pick the single most anomalous article and a typical comparator
most_anom = iso_news.nsmallest(1, 'anomaly_score').iloc[0]
case_cat = most_anom['category']

# Typical comparator: prefer a high-score non-anomalous article from the SAME category.
# If none exists, fall back to the highest-score article in that category, excluding the outlier.
typical_pool = iso_news[(iso_news['category'] == case_cat) & (~iso_news['is_anomaly'])]
if typical_pool.empty:
    typical_pool = iso_news[(iso_news['category'] == case_cat) & (iso_news.index != most_anom.name)]
if typical_pool.empty:
    typical_pool = iso_news[iso_news.index != most_anom.name]
typical = typical_pool.nlargest(1, 'anomaly_score').iloc[0]

def show_article(row, tag):
    abstract = str(row.get('abstract', ''))
    print(f"--- {tag} ---")
    print(f"Category:      {row['category']}")
    print(f"Title:         {row['title']}")
    print(f"Abstract:      {abstract[:280]}{'...' if len(abstract) > 280 else ''}")
    print(f"Length (words): {int(row['text_length'])}")
    print(f"Anomaly score:  {row['anomaly_score']:.4f}")
    print()

print(f"CASE STUDY — Most anomalous article vs typical article (target category: {case_cat})")
print("=" * 80)
print()
show_article(most_anom, "OUTLIER")
show_article(typical, "TYPICAL")


The case-study comparison shows that two articles with the same category label can differ sharply in texture and specificity. The typical article is short and formulaic. The anomalous article is longer, more feature-like, and anchored by specific people, places, or institutions.

**Interpretation.** The category label captures where the article belongs in the product taxonomy, but it does not fully capture article form. The outlier is not necessarily mislabeled; it may be a richer or more specific instance of a broad category.


## Q3B: Are anomalies tonally different?

Hypothesis: since outliers are short and factual (like the case study above), they should be tonally flatter than the longer explanatory articles. TextBlob gives a polarity score from -1 (negative) to +1 (positive); "flat" would mean close to zero.


In [ ]:
# Compute sentiment on the iso sample
sample_for_sentiment = iso_news.sample(min(2000, len(iso_news)), random_state=SEED)
sample_for_sentiment['sentiment'] = sample_for_sentiment['clean'].apply(
    lambda t: TextBlob(t).sentiment.polarity)

print("Mean sentiment polarity:")
print(f"  Normal articles:   {sample_for_sentiment[~sample_for_sentiment['is_anomaly']]['sentiment'].mean():.4f}")
print(f"  Anomalous articles:{sample_for_sentiment[sample_for_sentiment['is_anomaly']]['sentiment'].mean():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Sentiment distribution: normal vs anomaly
axes[0].hist(sample_for_sentiment[~sample_for_sentiment['is_anomaly']]['sentiment'],
             bins=40, alpha=0.6, density=True, color='steelblue', label='Normal')
axes[0].hist(sample_for_sentiment[sample_for_sentiment['is_anomaly']]['sentiment'],
             bins=40, alpha=0.6, density=True, color='tomato', label='Anomaly')
axes[0].set_xlabel("Sentiment Polarity"); axes[0].set_ylabel("Density")
axes[0].set_title("Sentiment: Normal vs. Anomalous Articles")
axes[0].legend()

# Sentiment by category
cat_sentiment = sample_for_sentiment.groupby('category')['sentiment'].mean().sort_values()
colors_sent   = ['tomato' if cat in anom_by_cat[anom_by_cat['Anomaly Rate'] > anom_by_cat['Anomaly Rate'].median()].index
                 else 'steelblue' for cat in cat_sentiment.index]
axes[1].barh(cat_sentiment.index, cat_sentiment.values, color=colors_sent)
axes[1].axvline(0, color='gray', linestyle='--')
axes[1].set_xlabel("Mean Sentiment Polarity")
axes[1].set_title("Sentiment by Category\n(red = high-anomaly categories)")

plt.tight_layout()
plt.show()

Mean sentiment polarity is similar for anomalous and normal articles, but the spread is narrower for anomalies. This does not support a simple claim that anomalies are more negative, more positive, or more neutral.

**Q3B conclusion.** Sentiment is not the main driver of anomaly status. The outlier signature appears to be more about specificity, length, and entity structure than about tone.


## Q3C: Do anomalies mention fewer entities?

Named entities (people, organizations, places, dates) are what thick articles use to anchor their stories. An article about "a new Lancet study on blood-pressure medication" will mention the journal, the drug, maybe a researcher. A thin wire brief just says "blood pressure drug helps" with nothing specific attached.

If anomalies are thin, they should have fewer entity mentions per article.


In [ ]:
# NER on a sample of anomalous vs normal articles
nlp = spacy.load("en_core_web_sm")

n_ner = 80
ano_texts = iso_news[iso_news['is_anomaly']]['text'].iloc[:n_ner].tolist()
norm_texts = iso_news[~iso_news['is_anomaly']]['text'].iloc[:n_ner].tolist()

def extract_entities(texts):
    ents = []
    for doc in nlp.pipe(texts, disable=['parser']):
        for ent in doc.ents:
            ents.append({'text': ent.text, 'label': ent.label_})
    # Always return the expected columns, even when no entities are found.
    return pd.DataFrame(ents, columns=['text', 'label'])

ano_ents = extract_entities(ano_texts)
norm_ents = extract_entities(norm_texts)

print("Entity type distribution — Normal articles:")
print(norm_ents['label'].value_counts().head(8).to_string() if len(norm_ents) else "No entities found.")
print("\nEntity type distribution — Anomalous articles:")
print(ano_ents['label'].value_counts().head(8).to_string() if len(ano_ents) else "No entities found.")


In [ ]:
# Visualise entity type profiles side by side
all_types = sorted(set(ano_ents['label'].dropna().unique()) | set(norm_ents['label'].dropna().unique()))

if len(all_types) == 0:
    print("No named entities found in either sample; skipping entity profile plot.")
else:
    norm_counts = norm_ents['label'].value_counts().reindex(all_types, fill_value=0)
    ano_counts = ano_ents['label'].value_counts().reindex(all_types, fill_value=0)

    # Normalise to rates (per article actually sampled)
    n_norm = max(1, len(norm_texts))
    n_ano = max(1, len(ano_texts))
    norm_rate = norm_counts / n_norm
    ano_rate = ano_counts / n_ano

    x = np.arange(len(all_types))
    w = 0.35

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(x - w/2, norm_rate, w, label='Normal', color='steelblue', alpha=0.8)
    ax.bar(x + w/2, ano_rate, w, label='Anomaly', color='tomato', alpha=0.8)
    ax.set_xticks(x); ax.set_xticklabels(all_types, rotation=45, ha='right')
    ax.set_ylabel("Named Entities per Article")
    ax.set_title("Entity Type Profile: Normal vs. Anomalous Articles\n"
                 "Do anomalies mention fewer or different kinds of entities?", fontsize=13)
    ax.legend()
    plt.tight_layout()
    plt.show()


Named-entity results challenge the original intuition. Rather than being entity-poor, anomalous articles contain more named entities across major entity types, especially people and organizations.

**Q3C conclusion.** Anomalies are often more information-dense than normal articles. Isolation Forest appears to flag articles with richer, more specific textual fingerprints, not merely articles with missing or generic content.


## Q3D: The outlier signature — summary table

Combining everything from Q3A–Q3C into one table. For each metric, compare the mean value in anomalous articles vs normal articles.


In [ ]:
# Merge in entity counts per article for the full iso sample
# (approximate: we have NER computed on small samples, so we summarize from those)

def safe_ratio(num, den):
    return np.nan if pd.isna(den) or den == 0 else num / den

summary_rows = []

# Length
summary_rows.append({
    'Metric': 'Text length (words)',
    'Normal': iso_news[~iso_news['is_anomaly']]['text_length'].mean(),
    'Anomaly': iso_news[ iso_news['is_anomaly']]['text_length'].mean()
})

# Sentiment polarity
sent_norm = sample_for_sentiment[~sample_for_sentiment['is_anomaly']]['sentiment']
sent_ano = sample_for_sentiment[ sample_for_sentiment['is_anomaly']]['sentiment']
summary_rows.append({
    'Metric': 'Sentiment polarity (mean)',
    'Normal': sent_norm.mean(),
    'Anomaly': sent_ano.mean()
})
summary_rows.append({
    'Metric': 'Sentiment polarity (std)',
    'Normal': sent_norm.std(),
    'Anomaly': sent_ano.std()
})

# Entities per article (from the NER sample)
n_norm = max(1, len(norm_texts))
n_ano = max(1, len(ano_texts))
summary_rows.append({
    'Metric': 'Entities per article (total)',
    'Normal': len(norm_ents) / n_norm,
    'Anomaly': len(ano_ents) / n_ano
})

# PERSON and ORG specifically
for lbl in ['PERSON', 'ORG']:
    summary_rows.append({
        'Metric': f'{lbl} mentions per article',
        'Normal': (norm_ents['label'] == lbl).sum() / n_norm,
        'Anomaly': (ano_ents['label'] == lbl).sum() / n_ano
    })

summary = pd.DataFrame(summary_rows)
summary['Ratio (Ano / Normal)'] = [safe_ratio(a, n) for a, n in zip(summary['Anomaly'], summary['Normal'])]
summary = summary.round(3)

print("THE OUTLIER SIGNATURE — one table, four signals")
print("=" * 70)
print(summary.to_string(index=False))
print()
print("Read this table descriptively: anomalous articles differ in length, entity density,")
print("and sentiment distribution from typical articles in this sample. Avoid hard-coding")
print("specific multipliers unless they match the values printed above.")


The summary table gives the final article-level signature: anomalous articles are generally **longer, richer in named entities, and tonally no more extreme than normal articles**.

---

### Q3 wrap-up

Q3 changes the interpretation of anomaly detection in this corpus. The anomalous articles are not primarily low-quality or underdeveloped examples. They are often detailed articles whose specificity makes them stand out from the short, repetitive, or formulaic articles that dominate some categories.

This matters for recommender systems. A statistical anomaly should not automatically be treated as irrelevant or mislabeled. It may instead deserve additional metadata, secondary category assignment, or embedding methods that better separate topical mismatch from article richness.


# Final Findings


## What defines a news category?

The results suggest that a news category is not a single statistical object. In MIND, a category appears to combine at least three layers:

1. **Topical domain.** Sports and parts of Finance behave this way: their vocabulary is distinctive and relatively compact.
2. **Editorial intent.** Labels such as News, Lifestyle, and Video often describe how content is packaged or consumed, not just what words appear in the article.
3. **Event-driven vocabulary.** News text is shaped by recurring people, places, institutions, and events. These can make articles look similar across categories or unusual within a category.

This explains the main pattern in the notebook. Categories can be meaningful to users while still failing to appear as clean clusters in TF-IDF space. The mismatch is not a contradiction; it reflects the difference between human editorial organization and machine-discovered text structure.


## Final conclusion

This project began with a plausible hypothesis: articles that do not fit their category might be bridge articles between categories. The evidence does not support that hypothesis as the main explanation.

The stronger conclusion is that anomalous articles are usually **peripheral, specific, and entity-rich articles inside broad editorial categories**. They sit away from category centers, but they do not consistently serve as bridges between clean category clusters. This is partly because the category space itself is already porous: many categories overlap, and some labels encode editorial purpose rather than a narrow topic.

The project therefore shows why outlier detection in news data must be interpreted carefully. A point far from its category centroid is not necessarily wrong. It may be a feature article inside a category dominated by short briefs, a locally specific article inside a broad label, or an article whose named entities create a distinctive lexical fingerprint.

In short: **news categories have shape, but the shape is soft-edged. The articles that do not fit are often not category mistakes; they are reminders that editorial labels are simpler than the stories they organize.**


## Note on the case-study article

The case-study article illustrates the broader result. Its anomalous status comes less from being obviously off-topic and more from being unusually detailed for its assigned category. It contains more specific references, a more feature-like structure, and a richer entity profile than a typical article in the same label.

This distinction is important. A recommender system that treats all anomalies as errors would miss precisely the articles that may be valuable because they are specific, contextual, or cross-cutting. Better systems should distinguish between **mislabeled content**, **bridge content**, and **rich peripheral content**.


## Future Work

Several extensions would strengthen this project:

1. **Use sentence embeddings.** Models such as SBERT or E5 could separate semantic similarity from surface word overlap better than TF-IDF.
2. **Model multi-label category membership.** Many articles plausibly belong to more than one category. A multi-label framework would better match the porous structure found here.
3. **Add temporal controls.** News vocabulary changes around major events. Temporal splits would show whether outliers are event-specific rather than structurally unusual.
4. **Compare anomaly detectors.** Isolation Forest should be compared with Local Outlier Factor, one-class SVMs, and embedding-distance methods.
5. **Evaluate with human judgment.** A small manual audit could distinguish true mislabels from rich peripheral articles and cross-category bridges.
6. **Separate article form from topic.** Length, entity count, and writing style should be modeled explicitly so that systems do not confuse richer writing with topical mismatch.


## Collaboration and Resource Declaration

On my honor, I declare the following resources:

- **Collaborators:** None.
- **Dataset / web sources:** MIND project page and Wu et al. (2020), MIND: A Large-scale Dataset for News Recommendation.
- **AI tools:** ChatGPT was used to help improve code robustness, polish writing, structure the notebook, and clarify interpretations. The analysis decisions, final review, and project responsibility remain my own.


In [ ]:
!pip freeze > requirements.txt
try:
    from google.colab import files
    files.download('requirements.txt')
except Exception:
    print("Saved requirements.txt in the current working directory.")
